In [ ]:
import torch
from transformer_lens import HookedTransformer
from transformer_lens.utils import test_prompt
import os, json
from typing import Union, Optional, Dict, Any, List, Tuple
from rich import print as rprint

In [ ]:
def new_test_prompt(
    prompt: str,
    answer: Union[str, list[str]],
    model,  # Can't give type hint due to circular imports
    #relation_name:str,
    #json_output_folder:str,
    prepend_space_to_answer: bool = True,
    print_details: bool = True,
    prepend_bos: Optional[bool] = False,
    top_k: int = 10,
) -> Dict[str, Any]:
    """
    Test if the Model Can Give the Correct Answer to a Prompt and calculate top-1 and top-10 accuracy.

    Args:
        Same as original function.

    Returns:
        A dictionary with:
        - top-1 and top-10 accuracy.
        - first_token_prob: Probability of the top-ranked token.
        - second_token_prob: Probability of the second-ranked token.
    """
    # os.makedirs(json_output_folder, exist_ok=True)
    # json_path = os.path.join(
    #     json_output_folder, relation_name.replace(" ", "_").lower() + ".json"
    # )
    answers = [answer] if isinstance(answer, str) else answer

    n_answers = len(answers)

    using_multiple_answers = n_answers > 1

    if prepend_space_to_answer:
        answers = [answer if answer.startswith(" ") else " " + answer for answer in answers]

    # Tokenize prompt and answer(s)
    prompt_tokens = model.to_tokens(prompt, prepend_bos=prepend_bos).to("cuda:0")
    answer_tokens = model.to_tokens(answers, prepend_bos=False).to("cuda:0")


    # If we have multiple answers, restrict to single-token answers
    if using_multiple_answers:
        answer_tokens = answer_tokens[:, :1]



    prompt_tokens = prompt_tokens.repeat(answer_tokens.shape[0], 1)
    tokens = torch.cat((prompt_tokens, answer_tokens), dim=1)

    prompt_str_tokens = model.to_str_tokens(prompt, prepend_bos=prepend_bos)

    answer_str_tokens_list = [model.to_str_tokens(answer, prepend_bos=False) for answer in answers]

    prompt_length = len(prompt_str_tokens)
    answer_length = 1 if using_multiple_answers else len(answer_str_tokens_list[0])


    if print_details:

        if using_multiple_answers:
            print("Tokenized answers:", answer_str_tokens_list)
        else:
            print("Tokenized answer:", answer_str_tokens_list[0])

    logits = model(tokens)
    probs = logits.softmax(dim=-1)

    # Initialize counters for top-1 and top-10 accuracy
    top_1_correct = 0
    top_10_correct = 0
    total_tokens = 0
    is_correct = False

    first_token_prob = None
    second_token_prob = None
    entries=[]

    for index in range(prompt_length, prompt_length + answer_length):
        # Get answer tokens for this sequence position
        answer_tokens = tokens[:, index]
        answer_str_tokens = [a[index - prompt_length] for a in answer_str_tokens_list]


        # Offset by 1 because models predict the NEXT token
        token_probs = probs[:, index - 1]
        sorted_token_probs, sorted_token_positions = token_probs.sort(descending=True)

        # Save the first and second token probabilities
        if first_token_prob is None:
            first_token_prob = sorted_token_probs[0, 0].item()
            second_token_prob = sorted_token_probs[0, 1].item()

        # Check correctness of the first answer token
        if index == prompt_length:  # Only check the first token
            predicted_token = model.to_string(sorted_token_positions[0, 0].item())

            actual_token = model.to_string(answer_tokens[0].item())
            is_correct = predicted_token == actual_token


        # Check if the answer token is in the top-1 or top-10 predictions
        for i, answer_token in enumerate(answer_tokens):
            #rank_tensor = (sorted_positions[0] == answer_tok).nonzero(as_tuple=True)[0]


            correct_token_rank = (sorted_token_positions[0] == answer_token).nonzero(as_tuple=True)[0]
            rank = correct_token_rank.item() if correct_token_rank.numel() > 0 else -1

            if correct_token_rank.numel() > 0:  # Token exists in predictions
                if correct_token_rank.item() == 0:  # Top-1
                    top_1_correct += 1
                if correct_token_rank.item() < top_k:  # Top-10
                    top_10_correct += 1

            top_k_predictions = []
            for j in range(top_k):
                top_k_predictions.append({
                    "rank":j,
                    "token": model.to_string(sorted_token_positions[0, j].item()),
                    "logit": logits[0, index - 1, sorted_token_positions[0, j]].item(),
                    "probability_percent": round(sorted_token_probs[0, j].item() * 100, 2)}
                )

            entries.append(
                {"prompt":prompt,
                 "answer_token": answer_str_tokens[i],

                "token_index": index - prompt_length,
                 "correct_rank": rank,
                 "logit": logits[0, index - 1, answer_token].item(),
                 "probability_percent": round(token_probs[0, answer_token].item() * 100, 2),
                 "is_top_1": int(correct_token_rank.item() == 0),
                 "is_top_10": int(0 <= correct_token_rank.item() < top_k),
                 "predicted_top1_token": model.to_string(sorted_token_positions[0, 0].item()),
                 "first_token_prob": first_token_prob,
                 "second_token_prob": second_token_prob,
                 "is_correct": is_correct,
                 "top_k_predictions": top_k_predictions
                 }
            )




        # Check if the top-ranked token matches the answer token
        #if model.to_string(answer_tokens[0]) == model.to_string(sorted_token_positions[0][0].item()):
        #    is_correct = True

        total_tokens += len(answer_tokens)


        if print_details:
            rprint(
                f"Performance on answer token{'s' if n_answers > 1 else ''}:\n"
                + "\n".join(
                    [
                        f"[b]Rank: {correct_token_rank.item(): <8} Logit: {logits[0, index-1, answer_tokens[i]].item():5.2f} "
                        f"Prob: {token_probs[0, answer_tokens[i]].item():6.2%} Token: |{answer_str_tokens[i]}|[/b]"
                        for i in range(len(answer_tokens))
                    ]
                )
            )
            for i in range(top_k):
                print(
                    f"Top {i}th token. Logit: {logits[0, index-1, sorted_token_positions[0, i]].item():5.2f} "
                    f"Prob: {sorted_token_probs[0, i].item():6.2%} Token: |{model.to_string(sorted_token_positions[0, i])}|"
                )

    # Calculate accuracies
    top_1_accuracy = (top_1_correct / total_tokens) * 100 if total_tokens > 0 else 0



    top_10_accuracy = (top_10_correct / total_tokens) * 100 if total_tokens > 0 else 0

    # if os.path.exists(json_path):
    #     with open(json_path, "r", encoding="utf-8") as f:
    #         existing = json.load(f)
    #         if not isinstance(existing, list):
    #             existing = []
    # else:
    #     existing = []

    # existing.extend(entries)

    # with open(json_path, "w", encoding="utf-8") as f:
    #     json.dump(existing, f, indent=2, ensure_ascii=False)

    # Return metrics, including token probabilities
    return {
        "top_1_accuracy": top_1_accuracy,
        "top_10_accuracy": top_10_accuracy,
        "first_token_prob": first_token_prob,
        "second_token_prob": second_token_prob,
        "is_correct": is_correct,
    }

In [ ]:
new_test_prompt("Die Vergangenheitsform von fragen ist", answer=' fragte', model=model, prepend_bos=False)

In [ ]:
new_test_prompt("The capital of Germany is:")

In [ ]:
!pip list

In [ ]:
def new_test_prompt(
    prompt: str,
    answer: Union[str, list[str]],
    model,  # Can't give type hint due to circular imports
    prepend_space_to_answer: bool = True,
    print_details: bool = True,
    prepend_bos: Optional[bool] = False,
    top_k: int = 10,
) -> Dict[str, Any]:
    """
    Test if the Model Can Give the Correct Answer to a Prompt and calculate top-1 and top-10 accuracy.

    Args:
        Same as original function.

    Returns:
        A dictionary with:
        - top-1 and top-10 accuracy.
        - first_token_prob: Probability of the top-ranked token.
        - second_token_prob: Probability of the second-ranked token.
    """
    answers = [answer] if isinstance(answer, str) else answer
    n_answers = len(answers)
    using_multiple_answers = n_answers > 1

    if prepend_space_to_answer:
        answers = [answer if answer.startswith(" ") else " " + answer for answer in answers]

    # Tokenize prompt and answer(s)
    prompt_tokens = model.to_tokens(prompt, prepend_bos=prepend_bos)
    answer_tokens = model.to_tokens(answers, prepend_bos=False)

    # If we have multiple answers, restrict to single-token answers
    if using_multiple_answers:
        answer_tokens = answer_tokens[:, :1]

    prompt_tokens = prompt_tokens.repeat(answer_tokens.shape[0], 1)
    tokens = torch.cat((prompt_tokens, answer_tokens), dim=1)

    prompt_str_tokens = model.to_str_tokens(prompt, prepend_bos=prepend_bos)
    answer_str_tokens_list = [model.to_str_tokens(answer, prepend_bos=False) for answer in answers]
    prompt_length = len(prompt_str_tokens)
    answer_length = 1 if using_multiple_answers else len(answer_str_tokens_list[0])

    if print_details:
        print("Tokenized prompt:", prompt_str_tokens)
        if using_multiple_answers:
            print("Tokenized answers:", answer_str_tokens_list)
        else:
            print("Tokenized answer:", answer_str_tokens_list[0])

    logits = model(tokens)
    probs = logits.softmax(dim=-1)

    # Initialize counters for top-1 and top-10 accuracy
    top_1_correct = 0
    top_10_correct = 0
    total_tokens = 0
    is_correct = False

    first_token_prob = None
    second_token_prob = None

    for index in range(prompt_length, prompt_length + answer_length):
        # Get answer tokens for this sequence position
        answer_tokens = tokens[:, index]
        answer_str_tokens = [a[index - prompt_length] for a in answer_str_tokens_list]

        # Offset by 1 because models predict the NEXT token
        token_probs = probs[:, index - 1]
        sorted_token_probs, sorted_token_positions = token_probs.sort(descending=True)

        # Save the first and second token probabilities
        if first_token_prob is None:
            first_token_prob = sorted_token_probs[0, 0].item()
            second_token_prob = sorted_token_probs[0, 1].item()
         
        # Check correctness of the first answer token
        if index == prompt_length:  # Only check the first token
            predicted_token = model.to_string(sorted_token_positions[0, 0].item())
            actual_token = model.to_string(answer_tokens[0].item())
            is_correct = predicted_token == actual_token

        # Check if the answer token is in the top-1 or top-10 predictions
        for i, answer_token in enumerate(answer_tokens):
            correct_token_rank = (sorted_token_positions[0] == answer_token).nonzero(as_tuple=True)[0]
            if correct_token_rank.numel() > 0:  # Token exists in predictions
                if correct_token_rank.item() == 0:  # Top-1
                    top_1_correct += 1
                if correct_token_rank.item() < top_k:  # Top-10
                    top_10_correct += 1


        # Check if the top-ranked token matches the answer token
        #if model.to_string(answer_tokens[0]) == model.to_string(sorted_token_positions[0][0].item()):
        #    is_correct = True

        total_tokens += len(answer_tokens)

        if print_details:
            rprint(
                f"Performance on answer token{'s' if n_answers > 1 else ''}:\n"
                + "\n".join(
                    [
                        f"[b]Rank: {correct_token_rank.item(): <8} Logit: {logits[0, index-1, answer_tokens[i]].item():5.2f} "
                        f"Prob: {token_probs[0, answer_tokens[i]].item():6.2%} Token: |{answer_str_tokens[i]}|[/b]"
                        for i in range(len(answer_tokens))
                    ]
                )
            )
            for i in range(top_k):
                print(
                    f"Top {i}th token. Logit: {logits[0, index-1, sorted_token_positions[0, i]].item():5.2f} "
                    f"Prob: {sorted_token_probs[0, i].item():6.2%} Token: |{model.to_string(sorted_token_positions[0, i])}|"
                )

    # Calculate accuracies
    top_1_accuracy = (top_1_correct / total_tokens) * 100 if total_tokens > 0 else 0
    top_10_accuracy = (top_10_correct / total_tokens) * 100 if total_tokens > 0 else 0

    # Return metrics, including token probabilities
    return {
        "top_1_accuracy": top_1_accuracy,
        "top_10_accuracy": top_10_accuracy,
        "first_token_prob": first_token_prob,
        "second_token_prob": second_token_prob,
        "is_correct": is_correct,
    }


In [ ]:
def load_models(model_name, device='cuda:7', checkpoint=""):
    model = HookedTransformer.from_pretrained_no_processing(
        model_name,
        dtype=torch.bfloat16,
        center_unembed=True,
        center_writing_weights=True,
        #fold_ln=True,
        device=device,
        trust_remote_code=True,
        cache_dir="/nfs/datx/hypersum/",
        #checkpoint_value=checkpoint,
    )

    return model


# Load the models
model_name =  "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:0', checkpoint="main")

In [ ]:
def load_json_files(directory: str, selected_category: str, selected_relation: str) -> Dict[str, List[Dict]]:
    """
    Load JSON files from the specified directory.

    Args:
        directory (str): Directory containing the JSON files.

    Returns:
        dict: Dictionary where keys are relation names and values are lists of data entries.
    """
    data = {}
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                path = os.path.join(root, file)
                with open(path, 'r') as f:
                    category = os.path.basename(root)
                    if category == selected_category or not selected_category:
                        relation = os.path.basename(file).replace('.json', '')
                        if relation == selected_relation or not selected_relation:
                            if relation not in data:
                                data[relation] = []
                            data[relation].append(json.load(f))
    return data


def parse_samples(data: Dict[str, List[Dict]]) -> Tuple[List[str], List[str]]:
    facts = []
    targets = []
    sentences = []
    subjects = []
    for entries in data:
        #for entry in entries:
        prompt_templates = entries['prompt_templates']
        samples = entries['samples']
        
        for sample in samples:
            subject = sample['subject']
            obj = " " + sample['object']
            
            
            #for template in prompt_templates:  # Iterate through multiple templates
                #fact = template.format(subject)
            subjects.append(subject)
            targets.append(obj)
            #sentences.append(fact + obj)
            # fact = prompt_templates.format(subject)
            # facts.append(fact)
            # targets.append(obj)
    return prompt_templates, subjects, targets

In [ ]:
relation_name = "person_father"

directory = "/mounts/data/proj/hypersum/Neuron_Analysis/data/relations_data"
data = load_json_files(directory, "factual", relation_name)
for key, value in data.items():
    #sentences, facts, targets = parse_samples(value)
    prompt_templates, subjects, targets = parse_samples(value)

In [ ]:
subjects

In [ ]:
# Cell 1 – Imports & helper functions

import os
import json
import requests
from collections import defaultdict
from tqdm import tqdm
import time


def load_json_files(directory: str,
                    selected_category: str = "",
                    selected_relation: str = "") -> Dict[str, List[Dict]]:
    """
    Walk `directory`, load all .json files whose parent‐dir == selected_category (or any
    if empty) and whose filename (sans .json) == selected_relation (or any if empty).
    Returns { relation_name: [parsed_json, …] }.
    """
    data: Dict[str, List[Dict]] = {}
    for root, _, files in os.walk(directory):
        category = os.path.basename(root)
        for file in files:
            if not file.endswith(".json"):
                continue
            rel_name = file[:-5]
            if (selected_category and category != selected_category) or \
               (selected_relation and rel_name != selected_relation):
                continue

            path = os.path.join(root, file)
            with open(path, 'r', encoding='utf-8') as f:
                payload = json.load(f)
            data.setdefault(rel_name, []).append(payload)
    return data

def parse_samples(entries: List[Dict]) -> Tuple[List[str], List[str], List[str]]:
    """
    entries: the list you got from load_json_files()[relation]
    Returns (prompt_templates, subjects, targets) where each target has a leading space.
    """
    prompt_templates: List[str] = []
    subjects: List[str] = []
    targets: List[str] = []

    for entry in entries:
        # each entry has its own set of templates + samples
        prompt_templates = entry['prompt_templates']
        for sample in entry['samples']:
            subjects.append(sample['subject'])
            # keep the leading space here; we'll strip later
            targets.append(" " + sample['object'])
    return prompt_templates, subjects, targets


def get_count(term: str,
              session: requests.Session = None,
              retries: int = 3,
              backoff_factor: float = 0.5) -> int:
    """
    Strip whitespace, ensure one leading space, then call Infini-Gram.
    Retries up to `retries` times with exponential back-off on failure.
    Returns int count or 0.
    """
    clean = term.strip()
    #query = clean if clean.startswith(" ") else " " + clean
    payload = {
        'index': 'v4_dolmasample_olmo',
        'query_type': 'count',
        'query': clean,
    }
    s = session or requests.Session()
    for attempt in range(1, retries + 1):
        try:
            resp = s.post('https://api.infini-gram.io/', json=payload, timeout=10)
            resp.raise_for_status()
            return int(resp.json().get('count', 0))
        except Exception as e:
            if attempt == retries:
                print(f"[Error] '{clean}' failed after {retries} attempts: {e}")
                return 0
            sleep_time = backoff_factor * (2 ** (attempt - 1))
            time.sleep(sleep_time)
    return 0

In [ ]:
# Cell 2 — Iterate all relations, fetch counts, save one output

# ── user params ─────────────────────────────────────────────
directory     = "/mounts/data/proj/hypersum/Neuron_Analysis/data/relations_data"
output_file   = "all_relations_counts.json"
# ─────────────────────────────────────────────────────────────

# 1) load _all_ relations
all_data = load_json_files(directory,
                           selected_category="semantic",     # empty = all categories
                           selected_relation="")     # empty = all relations

session = requests.Session()
results: Dict[str, Dict[str, Dict[str, int]]] = {}

# 2) loop
for relation, entries in tqdm(all_data.items(), desc="Relations"):
    _, subjects, targets = parse_samples(entries)

    # dedupe & sort for reproducibility
    uniq_subj = sorted(set(subjects))
    uniq_tgt  = sorted(set(targets))

    subj_counts   = {s.strip(): get_count(s, session) for s in uniq_subj}
    target_counts = {t.strip(): get_count(t, session) for t in uniq_tgt}

    results[relation] = {
        "subject_counts": subj_counts,
        "target_counts":  target_counts
    }

# 3) write one combined JSON
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ All done! Wrote counts for {len(results)} relations to {output_file}")


In [ ]:
def calculate_average_accuracy(data, model):
    total_top_1_accuracy = 0
    total_top_10_accuracy = 0
    total_examples = 0

    for key, value in data.items():
        # Extract sentences, prompts (facts), and answers (targets)
        prompt_templates, subjects, targets = parse_samples(value)

        # Loop through each fact and target pair
        for subj, target in zip(subjects, targets):
            for template in prompt_templates:
                # Format the fact using the template
                prompt = template.format(subj)
                
                # Evaluate the prompt
                result = new_test_prompt(
                    prompt=prompt,
                    answer=target,
                    model=model,
                    print_details=False  # Suppress detailed printing for bulk processing
                )
            
                # Accumulate the accuracies
                total_top_1_accuracy += result["top_1_accuracy"]
                total_top_10_accuracy += result["top_10_accuracy"]
                total_examples += 1

    # Calculate average accuracies
    average_top_1_accuracy = total_top_1_accuracy / total_examples
    average_top_10_accuracy = total_top_10_accuracy / total_examples

    return {
        "average_top_1_accuracy": average_top_1_accuracy,
        "average_top_10_accuracy": average_top_10_accuracy,
    }


In [ ]:
# Assume `data` is your dataset and `model` is your language model
average_accuracies = calculate_average_accuracy(data, model)

print("Average Top-1 Accuracy:", average_accuracies["average_top_1_accuracy"])
print("Average Top-10 Accuracy:", average_accuracies["average_top_10_accuracy"])

In [ ]:
def evaluate_templates_overall(
    subjects: List[str],
    targets: List[str],
    prompt_templates: List[str],
    model,
    second_token_threshold: float = 0.10
) -> Dict[str, Any]:
    """
    Evaluate all prompt templates and determine the best one based on overall performance,
    while identifying reliable and unreliable samples.

    Args:
        facts (list): List of facts (subject strings).
        targets (list): List of corresponding targets (answers).
        prompt_templates (list): List of prompt templates to test.
        model: The language model to evaluate.
        second_token_threshold (float): Maximum allowed probability for the second token.

    Returns:
        dict: A dictionary with:
            - template_scores: Overall scores and aggregated metrics for each template.
            - best_template: The best template based on the overall score.
            - best_template_metrics: Metrics for the best template.
            - reliable_samples: List of reliable samples with their details.
            - unreliable_samples: List of unreliable samples with their details.
    """
    template_scores = {template: {"first_token_prob_sum": 0, 
                                  "second_token_valid_count": 0,
                                  "total_count": 0} for template in prompt_templates}

    reliable_samples = []
    unreliable_samples = []

    for subject, target in zip(subjects, targets):
       
        sample_details = []

        for template in prompt_templates:
            # Format the fact using the template
            is_reliable = False
            prompt = template.format(subject)
            
            # Evaluate the prompt
            res = new_test_prompt(
                prompt=prompt,
                answer=target,
                model=model,
                print_details=True  # Suppress detailed printing for bulk processing
            )
            
            # Update metrics for the template
            if res["is_correct"]:
                template_scores[template]["first_token_prob_sum"] += res["first_token_prob"]
                template_scores[template]["total_count"] += 1
                if res["second_token_prob"] < second_token_threshold:
                    template_scores[template]["second_token_valid_count"] += 1
                    is_reliable = True
            else:
                template_scores[template]["first_token_prob_sum"] += 0
                template_scores[template]["total_count"] += 1
                
            # Collect sample details for each template
            sample_details.append({
                "template": template,
                "first_token_prob": res["first_token_prob"],
                "second_token_prob": res["second_token_prob"],
                "is_reliable": res["second_token_prob"] < second_token_threshold,
            })

            # Add sample to reliable or unreliable list
            if is_reliable:
                reliable_samples.append({"subject": subject, "object": target.strip()})
            else:
                unreliable_samples.append({"subject": subject, "object": target.strip()})

    # Calculate average scores and reliability for each template
    overall_scores = {}
    for template, metrics in template_scores.items():
        avg_first_token_prob = metrics["first_token_prob_sum"] / metrics["total_count"]
        reliability = metrics["second_token_valid_count"] / metrics["total_count"]
        overall_scores[template] = {
            "average_first_token_prob": avg_first_token_prob,
            "reliability": reliability,
            "overall_score": avg_first_token_prob * reliability,  # Combined score
        }

    # Find the best template based on the overall score
    best_template = max(overall_scores.items(), key=lambda x: x[1]["overall_score"])

    return {
        "template_scores": overall_scores,
        "best_template": best_template[0],
        "best_template_metrics": best_template[1],
        "reliable_samples": reliable_samples,
        "unreliable_samples": unreliable_samples,
    }


In [ ]:
result = evaluate_templates_overall(subjects, targets, prompt_templates, model, second_token_threshold=0.10)

# Display results
print("Template Scores:")
for template, metrics in result["template_scores"].items():
    print(f"Template: {template}")
    print(f"  Average First Token Prob: {metrics['average_first_token_prob']:.2f}")
    print(f"  Reliability: {metrics['reliability']:.2%}")
    print(f"  Overall Score: {metrics['overall_score']:.2f}")
    print("-" * 50)

print(f"Best Template: {result['best_template']}")
print(f"Best Template Metrics: {result['best_template_metrics']}")

In [ ]:
def evaluate_best_template_samples(
    subjects: List[str],
    targets: List[str],
    best_template: str,
    model,
    first_token_threshold = 0.75,
    second_token_threshold: float = 0.10
) -> Dict[str, List[Dict]]:
    """
    Evaluate reliable and unreliable samples using the best template.

    Args:
        facts (list): List of facts (subject strings).
        targets (list): List of corresponding targets (answers).
        best_template (str): The best template determined earlier.
        model: The language model to evaluate.
        second_token_threshold (float): Maximum allowed probability for the second token.

    Returns:
        dict: A dictionary with:
            - reliable_samples: List of reliable samples in the specified format.
            - unreliable_samples: List of unreliable samples in the specified format.
    """
    reliable_samples = []
    unreliable_samples = []

    for subject, target in zip(subjects, targets):
        # Format the fact using the best template
        prompt = best_template.format(subject)
        
        # Evaluate the prompt
        res = new_test_prompt(
            prompt=prompt,
            answer=target,
            model=model,
            print_details=False  # Suppress detailed printing for bulk processing
        )
        
        # Determine reliability based on second token probability
        if res['is_correct'] == True and res["first_token_prob"] > first_token_threshold and res["second_token_prob"] < second_token_threshold:
            reliable_samples.append({"subject": subject, "object": target.strip()})
        else:
            unreliable_samples.append({"subject": subject, "object": target.strip()})

    return {
        "reliable_samples": reliable_samples,
        "unreliable_samples": unreliable_samples,
    }

In [ ]:
best_template = result["best_template"]
print(f"Best Template: {best_template}")

# Evaluate reliable and unreliable samples for the best template
sample_evaluation_result = evaluate_best_template_samples(
    subjects=subjects,
    targets=targets,
    best_template=best_template,
    model=model,
    first_token_threshold = 0.75,
    second_token_threshold=0.10
)

In [ ]:
print(f"Reliable Examples: {sample_evaluation_result['reliable_samples']}")
print(f"Unreliable Examples: {sample_evaluation_result['unreliable_samples']}")

In [ ]:
import numpy as np
t1 = list(np.random.choice(sample_evaluation_result['reliable_samples'], 20, replace=False))

In [ ]:
sample_evaluation_result['reliable_samples']

In [ ]:
sample_evaluation_result['unreliable_samples']

In [ ]:
def generate_output_file(
    name: str,
    best_template: str,
    samples: List[Dict[str, str]],
    output_file: str
):
    """
    Generate an output JSON file using the best template.

    Args:
        name (str): The name of the relation.
        best_template (str): The best performing prompt template.
        output_file (str): The file path for saving the output JSON.
    """
    output_data = {
        "name": name,
        "prompt_templates": [best_template],
        "samples": samples
    }

    # Write the JSON to the output file
    with open(output_file, "w") as f:
        json.dump(output_data, f, indent=4)

    print(f"Output file generated at: {output_file}")


In [ ]:
output_file_path = "/nfs/datx/hypersum/Neuron_Analysis/data/relations_data/semantic"
best_template = result['best_template']

generate_output_file(
    name=relation_name,
    best_template=best_template,
    samples = sample_evaluation_result['reliable_samples'],
    output_file=f"{output_file_path}/{relation_name}.json"
)

### End of the Notebook after that are only test cases

In [ ]:
new_test_prompt("Santiago de Chile is part of the country of", answer='Chile', model=model, prepend_bos=False)

In [ ]:
new_test_prompt("Who is the CEO of Raytheon Company? Their name is", answer='Thomas Kennedy', model=model, prepend_bos=False)

In [ ]:
model.to_str_tokens('Movie')

In [ ]:
new_test_prompt("The Movie Forrest Gump was directed by the director with the name of", answer="Robert Zemeckis", model=model, prepend_bos=False)

In [ ]:
new_test_prompt('The headquarters of Autodesk Media and Entertainment are in the city of', answer='Montreal', model=model, prepend_bos=False)

In [ ]:
new_test_prompt("Wii Balance Board is a product by the company of", answer=' Nintendo', model=model, prepend_bos=False)

In [ ]:
new_test_prompt("What is the capital city of Turkey? The capital city is", answer=' Ankara', model=model, prepend_bos=False)

In [ ]:
test_prompt("Lebron James plays in the league called the", answer=' NBA', model=model, prepend_bos=False)

In [ ]:
test_prompt("The Colosseum is located in the city of", answer='Rome', model=model, prepend_bos=False)